# 12 — Embedding baselines (matched protocol)

Dr. Vita's original comparison (`Comparison-CZ_bin_classification_multi_encoder_colab.ipynb`)
trained five Czech/multilingual transformer encoders on 327 documents (`train` +
`train.extra`), used a single 75/25 train/val split (`SEED=1704`), ran 8 epochs with no
repeated CV, and reported point-estimate test metrics with no bootstrap CIs.

This notebook reruns the **same five checkpoints** under the **exact evaluation protocol
of the frozen V2 pipeline** (notebooks 09/10): 241 labelled train docs only, 10× repeated
stratified 5-fold CV (`SEED=42`), paired bootstrap contrasts, and a one-shot 61-doc test
evaluation. The result is a comparison table where every row was produced the same way —
the asymmetry between "new embedding numbers" and "old LLM-feature numbers" is gone.

Two approaches per model:

* **(a) Frozen encoder + `LogisticRegressionCV` probe** — always runs. [CLS] (or mean-pool)
  embeddings extracted once, cached to disk, then a linear probe under the matched CV.
* **(b) Fine-tuned classifier** — gated by `RUN_FINETUNE`. Fine-tunes a classification head
  end-to-end *inside* CV folds (never touching that fold's held-out validation rows or the
  test set), with early stopping on fold-internal validation loss. The notebook's matched
  protocol is 10× repeated stratified 5-fold (50 folds); fine-tuning 5 transformer models
  over 50 folds each is impractical on a single machine, so this arm deliberately runs a
  **lighter, separately-labelled single stratified 5-fold CV** instead (5 fold-fits per
  model, not 50) — still 241 docs only, still proper held-out CV, just not repeated. See
  Block 7 for the full reasoning and the overfitting safeguards this requires.

`train.extra` / `overview/` is never read here — only the 241 labelled train docs and the
61 sealed test docs, same as notebooks 09/10. `frozen_spec.json` is read-only.

## Block 0 — Configuration, pinned versions, device

In [1]:
import json, re, time, warnings, datetime
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

from helpers import SEED, N_REPEATS_DEFAULT, N_SPLITS_DEFAULT, cv_proba, boot_ci, paired_bootstrap

# --- library versions are pinned here; a mismatch is a loud warning, not a crash, since
# the frozen-encoder probe arm only needs torch+transformers to run a forward pass. ---
EXPECTED_VERSIONS = {"torch": "2.9.1", "transformers": "5.17.0"}
import torch
import transformers
_got = {"torch": torch.__version__, "transformers": transformers.__version__}
for _lib, _exp in EXPECTED_VERSIONS.items():
    if _got[_lib] != _exp:
        print(f"WARNING: {_lib} {_got[_lib]} != pinned {_exp} -- results should still hold, "
              f"but a library upgrade is the first thing to check if numbers drift.")
print("library versions:", _got)

RUN_FINETUNE = True   # (b) is a lighter, separately-labelled single 5-fold CV (not the
                       # notebook's matched 10x5) -- 5 fold-fits per model instead of 50.
                       # Left OFF on THIS machine: fine-tuning two different transformer
                       # architectures back-to-back in one process was found to reliably
                       # hang the MPS backend (confirmed via isolated repro; not a timing
                       # issue -- the process sits in an uninterruptible kernel wait and
                       # never returns). Block 7 isolates each model in its own subprocess
                       # (_ft_worker.py) as a mitigation, but this machine's ~8GB RAM makes
                       # even that arm slow/fragile. Flip to True on a machine with a real
                       # GPU (CUDA) or more headroom; see Block 7 for how to run it there.

DATA       = Path("fileDataset")
OUT        = DATA / "outputs"
FROZEN_DIR = OUT / "frozen_pipeline_v2"
assert FROZEN_DIR.exists(), f"{FROZEN_DIR} not found -- run notebooks 08-10 first"

device = torch.device("cuda" if torch.cuda.is_available() else
                       ("mps" if torch.backends.mps.is_available() else "cpu"))
HAS_ACCELERATOR = device.type in ("cuda", "mps")
print(f"device        : {device}  (accelerator available: {HAS_ACCELERATOR})")
print(f"RUN_FINETUNE  : {RUN_FINETUNE}")
print(f"CV protocol   : {N_REPEATS_DEFAULT}x repeated stratified {N_SPLITS_DEFAULT}-fold, SEED={SEED}")

library versions: {'torch': '2.9.1+cu128', 'transformers': '5.17.0'}
device        : cuda  (accelerator available: True)
RUN_FINETUNE  : True
CV protocol   : 10x repeated stratified 5-fold, SEED=42


## Block 1 — Frozen V2 spec (read-only) + Dr. Vita's unmatched reference panel

`frozen_spec.json` is loaded for its CV/test numbers only — **never written to**. Dr.
Vita's original numbers are hardcoded from the reference notebook and printed in a clearly
separated panel: same checkpoints, unmatched protocol (327 docs, single 75/25 split,
`SEED=1704`, no CV, no bootstrap). They are context, not a comparison.

In [2]:
spec = json.loads((FROZEN_DIR / "frozen_spec.json").read_text())
FROZEN_PRIMARY_FEATURES = spec["feature_sets"]["frozen_primary"]
V2_STABLE               = spec["feature_sets"]["V2_STABLE"]
SURFACE                 = spec["feature_sets"]["SURFACE"]

print("Frozen V2 pipeline (read-only, from frozen_spec.json):")
print(f"  V2 stable+surface   CV AUC = {spec['frozen_primary_cv_auc']:.4f}")
print(f"  V2 stable+surface  test AUC = {spec['test_results']['primary_auc']:.4f}"
      f"  95% CI {spec['test_results']['primary_ci95']}")
print(f"  surface baseline    test AUC = {spec['test_results']['surface_auc']:.4f}")

# Dr. Vita's original comparison -- 327 docs (train + train.extra), single 75/25 split
# (SEED=1704), 8 epochs, no repeated CV, no bootstrap CI. NOT comparable to the matched
# table this notebook builds -- printed here only so a reader sees both numbers.
ORIGINAL_UNMATCHED = pd.DataFrame([
    {"model": "C5-FERNET",     "hf_checkpoint": "fav-kky/FERNET-C5-RoBERTa",             "original_test_auc": 0.744},
    {"model": "RobeCzech",     "hf_checkpoint": "ufal/robeczech-base",                    "original_test_auc": 0.836},
    {"model": "CZERT-B",       "hf_checkpoint": "UWB-AIR/Czert-B-base-cased",             "original_test_auc": 0.822},
    {"model": "mBERT",         "hf_checkpoint": "google-bert/bert-base-multilingual-cased","original_test_auc": 0.798},
    {"model": "Small-E-Czech", "hf_checkpoint": "Seznam/small-e-czech",                   "original_test_auc": 0.782},
])
print("\n" + "=" * 78)
print("UNMATCHED REFERENCE -- Dr. Vita's original notebook (327 docs, single 75/25 split,")
print("SEED=1704, 8 epochs, no repeated CV, no bootstrap CI). NOT comparable to the")
print("matched table below -- context only.")
print("=" * 78)
print(ORIGINAL_UNMATCHED.to_string(index=False))
print("=" * 78)

Frozen V2 pipeline (read-only, from frozen_spec.json):
  V2 stable+surface   CV AUC = 0.8058
  V2 stable+surface  test AUC = 0.8070  95% CI [0.6768258298184486, 0.9159444807671352]
  surface baseline    test AUC = 0.7957

UNMATCHED REFERENCE -- Dr. Vita's original notebook (327 docs, single 75/25 split,
SEED=1704, 8 epochs, no repeated CV, no bootstrap CI). NOT comparable to the
matched table below -- context only.
        model                            hf_checkpoint  original_test_auc
    C5-FERNET                fav-kky/FERNET-C5-RoBERTa              0.744
    RobeCzech                      ufal/robeczech-base              0.836
      CZERT-B               UWB-AIR/Czert-B-base-cased              0.822
        mBERT google-bert/bert-base-multilingual-cased              0.798
Small-E-Czech                     Seznam/small-e-czech              0.782


## Block 2 — Load transcripts (241 train / 61 test)

Identical `read_split` / `surface()` as notebooks 09/10. Train docs are sorted by filename
to reproduce notebook 09's exact `sorted(common_lab)` document order — this is what makes
the CV fold assignment (Block 3) byte-identical to the frozen pipeline's, which is required
for the paired bootstrap in Block 8 to be a legitimate *matched-fold* comparison. `overview/`
(`train.extra`) is never read.

In [3]:
def read_split(folder, label):
    return [{"file": f.name, "label": label,
             "text": f.read_text(encoding="utf-8", errors="replace").strip()}
            for f in sorted(folder.glob("*.txt"))]

_WORD = re.compile(r"\w+", re.UNICODE)
def surface(t):
    nw = max(1, len(t.split()))
    nt = len(set(_WORD.findall(t.lower())))
    ns = max(1, len(re.findall(r"[.!?]+", t)))
    return {"n_char": len(t), "n_word": nw, "n_type": nt, "ttr": nt / nw,
            "n_sent": ns, "mlu": nw / ns, "n_comma": t.count(",")}

train_docs = pd.DataFrame(read_split(DATA / "train" / "negative", 0) +
                           read_split(DATA / "train" / "positive", 1))
train_docs = pd.concat([train_docs, train_docs.text.apply(lambda t: pd.Series(surface(t)))], axis=1)
train_docs = train_docs.sort_values("file").reset_index(drop=True)   # == sorted(common_lab) in nb09
assert len(train_docs) == 241

test_docs = pd.DataFrame(read_split(DATA / "test" / "negative", 0) +
                          read_split(DATA / "test" / "positive", 1))
test_docs = pd.concat([test_docs, test_docs.text.apply(lambda t: pd.Series(surface(t)))], axis=1)
assert len(test_docs) == 61

y_train = train_docs.label.values.astype(int)
y_test  = test_docs.label.values.astype(int)
texts_train = train_docs.text.tolist()
texts_test  = test_docs.text.tolist()

print(f"train: {len(train_docs)} docs (pos {y_train.sum()}, neg {(y_train == 0).sum()})"
      f"  median words {train_docs.n_word.median():.0f}")
print(f"test : {len(test_docs)} docs (pos {y_test.sum()}, neg {(y_test == 0).sum()})"
      f"  median words {test_docs.n_word.median():.0f}")

train: 241 docs (pos 70, neg 171)  median words 70
test : 61 docs (pos 19, neg 42)  median words 65


## Block 3 — Matched-protocol predictions for the existing baselines

Per-document out-of-fold probabilities for the surface / V2-frozen-primary / TF-IDF arms
were never persisted by notebook 09 (only aggregate CV metrics were saved, to
`v2_stability_comparison.csv`) — only the *fitted-on-all-241* pipelines were frozen
(`pipe_*.joblib`), for the one-shot test evaluation. There is therefore no on-disk OOF
array to "load" for the CV paired bootstrap.

What this block does instead, in two halves:

* **CV arm** — rebuild the exact same feature matrices (from `v2_raw.jsonl`, already
  extracted by notebook 08 — no new LLM calls) and reproduce the OOF predictions with
  `cv_proba` using the **frozen classifier configs verbatim** (elastic-net for
  surface/V2, L2 for TF-IDF — copied from notebook 09, not redesigned). Because the
  document order above matches notebook 09's exactly, the fold assignment is
  byte-identical and the reproduced AUC is checked against the recorded value below —
  this is a reproducibility check, not a new experiment.
* **Test arm** — load the already-fitted `pipe_surface.joblib` / `pipe_v2_stable_surface.joblib`
  artifacts and call `.predict_proba()` only. No fitting happens here at all.

In [4]:
LIST_FIELDS = {"n_entities": "named_entities", "n_specific_verb": "specific_action_verbs",
               "n_generic_verb": "generic_verbs", "n_locative": "locative_expressions",
               "n_hedge": "hedge_spans", "n_deictic": "deictic_spans",
               "n_metacomment": "metacomment_spans", "n_diminutive": "diminutive_or_affective_forms",
               "n_quantity": "quantity_expressions"}
INT_FIELDS = {"n_proposition": "complete_propositions", "n_selfcorrect": "self_corrections"}

def to_row(obj):
    '''Identical to the to_row() in notebooks 08/09/10 -- raw LLM JSON -> count columns.'''
    r = {}
    for out, key in LIST_FIELDS.items():
        v = obj.get(key) or []
        r[out] = len(v) if isinstance(v, list) else 0
    for out, key in INT_FIELDS.items():
        v = obj.get(key, 0)
        r[out] = int(v) if isinstance(v, (int, float)) else 0
    reg = obj.get("regions_referenced") or []
    r["n_region"] = len(set(reg)) if isinstance(reg, list) else 0
    rep = obj.get("repeated_content_lemmas") or []
    r["n_repeated_lemma"] = sum(1 for x in rep if isinstance(x, dict) and x.get("lemma"))
    r["repeat_mass"] = sum(int(x.get("count", 0)) for x in rep
                           if isinstance(x, dict) and str(x.get("count", "")).isdigit())
    return r

def load_raw(path):
    out = {}
    for line in path.read_text(encoding="utf-8").splitlines():
        if line.strip():
            rec = json.loads(line)
            out[rec["file"]] = rec
    return out

def build_features(docs, raw_path):
    raw = load_raw(raw_path)
    rows = [{"file": f, **to_row(rec["response"])}
            for f, rec in raw.items() if rec.get("response") is not None]
    feat = pd.DataFrame(rows)
    counts = [c for c in feat.columns if c != "file"]
    d = docs.merge(feat, on="file", how="inner")
    for c in counts:
        d[c + "_r100"] = 100 * d[c] / d["n_word"].clip(lower=1)
    d["region_breadth"] = d.n_region / 3.0
    return d

dfA = build_features(train_docs, OUT / "v2_raw.jsonl")
dfA = dfA.sort_values("file").reset_index(drop=True)
assert len(dfA) == 241 and (dfA.file.values == train_docs.file.values).all()

dfT = build_features(test_docs, OUT / "v2_test_raw.jsonl")
assert len(dfT) == 61 and (dfT.file.values == test_docs.file.values).all()

print(f"rebuilt V2+surface feature matrix: train {dfA.shape}  test {dfT.shape}")

rebuilt V2+surface feature matrix: train (241, 39)  test (61, 39)


In [5]:
from sklearn.linear_model import LogisticRegression, LogisticRegressionCV
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, balanced_accuracy_score, f1_score

# classifier factories copied verbatim from notebook 09 Block 9/10 -- not redesigned.
def enet():
    return make_pipeline(StandardScaler(), LogisticRegressionCV(
        penalty="elasticnet", solver="saga", l1_ratios=[0.3, 0.6, 0.9], Cs=8,
        max_iter=5000, class_weight="balanced", scoring="roc_auc",
        cv=StratifiedKFold(5, shuffle=True, random_state=SEED), n_jobs=-1))

def tfidf_l2():
    return make_pipeline(
        TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=2, sublinear_tf=True),
        LogisticRegression(max_iter=3000, class_weight="balanced"))

X_surf_train = dfA[SURFACE].values
X_v2fp_train = dfA[FROZEN_PRIMARY_FEATURES].values
X_text_train = np.array(dfA.text.tolist(), dtype=object)

BASELINE_CV_CACHE = OUT / "embedding_baseline_reproduced_oof.npz"
if BASELINE_CV_CACHE.exists():
    _c = np.load(BASELINE_CV_CACHE)
    p_surf, p_v2fp, p_tfidf = _c["surf"], _c["v2fp"], _c["tfidf"]
    print(f"loaded cached reproduced OOF predictions from {BASELINE_CV_CACHE.name}")
else:
    t0 = time.time()
    p_surf  = cv_proba(enet(), X_surf_train, y_train, seed=SEED)
    p_v2fp  = cv_proba(enet(), X_v2fp_train, y_train, seed=SEED)
    p_tfidf = cv_proba(tfidf_l2(), X_text_train, y_train, seed=SEED)
    np.savez(BASELINE_CV_CACHE, surf=p_surf, v2fp=p_v2fp, tfidf=p_tfidf)
    print(f"reproduced + cached baseline OOF predictions  [{time.time() - t0:.0f}s]")

# reproducibility guard -- these must match notebook 09's recorded numbers (same fold
# assignment, same classifier config) to a tight tolerance, or the document order above
# has drifted from notebook 09's and the paired bootstrap below would not be trustworthy.
_recorded = pd.read_csv(OUT / "v2_stability_comparison.csv").set_index("key")
for _key, _p in [("surf", p_surf), ("v2fp", p_v2fp), ("tfidf", p_tfidf)]:
    _fresh = roc_auc_score(y_train, _p)
    _ref   = _recorded.loc[_key, "AUC"]
    print(f"  {_key:6s}: reproduced AUC {_fresh:.4f}  vs recorded {_ref:.4f}"
          f"  (Δ={_fresh - _ref:+.4f})")
    assert abs(_fresh - _ref) < 0.01, f"{_key} reproduction drifted from notebook 09 -- STOP"
print("reproduction verified against notebook 09's recorded numbers -- safe to use for pairing.")

loaded cached reproduced OOF predictions from embedding_baseline_reproduced_oof.npz
  surf  : reproduced AUC 0.7427  vs recorded 0.7428  (Δ=-0.0001)
  v2fp  : reproduced AUC 0.8058  vs recorded 0.8058  (Δ=+0.0000)
  tfidf : reproduced AUC 0.8673  vs recorded 0.8673  (Δ=+0.0000)
reproduction verified against notebook 09's recorded numbers -- safe to use for pairing.


In [6]:
import joblib

art_surface = joblib.load(FROZEN_DIR / "pipe_surface.joblib")
art_v2fp    = joblib.load(FROZEN_DIR / "pipe_v2_stable_surface.joblib")

# pure forward pass on already-fitted, frozen pipelines -- no fitting happens here.
proba_surface_test = art_surface["pipeline"].predict_proba(dfT[SURFACE].values)[:, 1]
proba_v2fp_test    = art_v2fp["pipeline"].predict_proba(dfT[FROZEN_PRIMARY_FEATURES].values)[:, 1]

print(f"surface (frozen) test AUC = {roc_auc_score(y_test, proba_surface_test):.4f}"
      f"  (spec: {spec['test_results']['surface_auc']:.4f})")
print(f"V2 fp   (frozen) test AUC = {roc_auc_score(y_test, proba_v2fp_test):.4f}"
      f"  (spec: {spec['test_results']['primary_auc']:.4f})")

surface (frozen) test AUC = 0.7957  (spec: 0.7957)
V2 fp   (frozen) test AUC = 0.8070  (spec: 0.8070)


## Block 4 — Embedding model registry

Same five HuggingFace checkpoints as Dr. Vita's original notebook. [CLS] (first-token)
pooling is used for the four BERT/RoBERTa-family encoders — the same pooling their
`AutoModelForSequenceClassification` heads use internally. Small-E-Czech is an ELECTRA
discriminator, which was never pretrained with a CLS-style sentence objective, so it is
mean-pooled over the attention mask instead (standard practice for ELECTRA embeddings).

In [7]:
MODEL_CONFIGS = [
    {"key": "fernet_c5_roberta",   "display_name": "C5-FERNET (frozen)",     "hf": "fav-kky/FERNET-C5-RoBERTa",              "pooling": "cls"},
    {"key": "robeczech_base",      "display_name": "RobeCzech (frozen)",     "hf": "ufal/robeczech-base",                    "pooling": "cls"},
    {"key": "czert_b_base_cased",  "display_name": "CZERT-B (frozen)",       "hf": "UWB-AIR/Czert-B-base-cased",             "pooling": "cls"},
    {"key": "mbert_base_cased",    "display_name": "mBERT (frozen)",         "hf": "google-bert/bert-base-multilingual-cased","pooling": "cls"},
    {"key": "small_e_czech",       "display_name": "Small-E-Czech (frozen)", "hf": "Seznam/small-e-czech",                   "pooling": "mean"},
]

MAX_LENGTH = 256   # matches the original notebook's MAX_LENGTH, reused for the fine-tune arm too

for m in MODEL_CONFIGS:
    print(f"  {m['key']:20s} {m['hf']:45s} pooling={m['pooling']}")

  fernet_c5_roberta    fav-kky/FERNET-C5-RoBERTa                     pooling=cls
  robeczech_base       ufal/robeczech-base                           pooling=cls
  czert_b_base_cased   UWB-AIR/Czert-B-base-cased                    pooling=cls
  mbert_base_cased     google-bert/bert-base-multilingual-cased      pooling=cls
  small_e_czech        Seznam/small-e-czech                          pooling=mean


## Block 5 — Frozen-encoder embedding extraction (cached)

One forward pass per document, no gradient, no fine-tuning. Embeddings are cached to
`.npy` so re-running this notebook after Block 3/6 changes skips extraction entirely.

In [8]:
from transformers import AutoTokenizer, AutoModel

@torch.no_grad()
def embed_texts(hf_name, texts, pooling, batch_size=16, max_length=MAX_LENGTH):
    tok   = AutoTokenizer.from_pretrained(hf_name)
    model = AutoModel.from_pretrained(hf_name).to(device).eval()
    vecs = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        enc = tok(batch, padding=True, truncation=True, max_length=max_length,
                  return_tensors="pt").to(device)
        hidden = model(**enc).last_hidden_state
        if pooling == "cls":
            v = hidden[:, 0, :]
        else:
            mask = enc["attention_mask"].unsqueeze(-1).float()
            v = (hidden * mask).sum(1) / mask.sum(1).clamp(min=1e-9)
        vecs.append(v.float().cpu().numpy())
    del model
    if device.type == "mps":
        torch.mps.empty_cache()
    elif device.type == "cuda":
        torch.cuda.empty_cache()
    return np.concatenate(vecs, axis=0)


def get_or_build_embeddings(model_cfg):
    key = model_cfg["key"]
    tr_path = OUT / f"{key}_embeddings_train.npy"
    te_path = OUT / f"{key}_embeddings_test.npy"
    if tr_path.exists() and te_path.exists():
        return np.load(tr_path), np.load(te_path)
    t0 = time.time()
    Xtr = embed_texts(model_cfg["hf"], texts_train, model_cfg["pooling"])
    Xte = embed_texts(model_cfg["hf"], texts_test, model_cfg["pooling"])
    np.save(tr_path, Xtr)
    np.save(te_path, Xte)
    print(f"  {key}: extracted + cached  train {Xtr.shape}  test {Xte.shape}  [{time.time() - t0:.0f}s]")
    return Xtr, Xte

EMBEDDINGS = {}
print("extracting / loading cached embeddings for 5 frozen encoders...")
for m in MODEL_CONFIGS:
    try:
        EMBEDDINGS[m["key"]] = get_or_build_embeddings(m)
    except Exception as e:
        print(f"  {m['key']}: FAILED to load/extract ({type(e).__name__}: {e}) -- skipping this model")
print(f"embeddings ready for: {list(EMBEDDINGS)}")

extracting / loading cached embeddings for 5 frozen encoders...
embeddings ready for: ['fernet_c5_roberta', 'robeczech_base', 'czert_b_base_cased', 'mbert_base_cased', 'small_e_czech']


## Block 6 — Frozen encoder + `LogisticRegressionCV` probe: matched CV + one-shot test

Same protocol as every other arm in this project: `cv_proba` (10×5 repeated stratified,
`SEED=42`) for the CV estimate, then a single fit on all 241 train embeddings and a
`predict_proba` on the 61 test embeddings for the one-shot test estimate. The probe is
`StandardScaler` + `LogisticRegressionCV` (L2, `Cs` tuned by inner 5-fold CV) — the same
family of classifier as the surface baseline's robustness arm, applied here as primary
since these are dense, high-dimensional embeddings rather than a handful of engineered
features.

In [9]:
def probe_clf():
    return make_pipeline(StandardScaler(), LogisticRegressionCV(
        Cs=10, penalty="l2", solver="lbfgs", max_iter=5000, class_weight="balanced",
        scoring="roc_auc", cv=StratifiedKFold(5, shuffle=True, random_state=SEED), n_jobs=-1))

PROBA_FROZEN, TEST_PROBA_FROZEN, FROZEN_RESULTS = {}, {}, []

for m in MODEL_CONFIGS:
    key = m["key"]
    if key not in EMBEDDINGS:
        continue
    Xtr, Xte = EMBEDDINGS[key]
    t0 = time.time()
    p_cv = cv_proba(probe_clf(), Xtr, y_train, seed=SEED)
    ci_lo, ci_hi = boot_ci(y_train, p_cv, seed=SEED)

    clf = probe_clf().fit(Xtr, y_train)
    p_test = clf.predict_proba(Xte)[:, 1]
    ci_lo_t, ci_hi_t = boot_ci(y_test, p_test, seed=SEED)

    PROBA_FROZEN[key] = p_cv
    TEST_PROBA_FROZEN[key] = p_test
    FROZEN_RESULTS.append({
        "key": key, "representation": m["display_name"], "type": "embedding",
        "n_feat": Xtr.shape[1],
        "cv_auc": roc_auc_score(y_train, p_cv), "cv_ci_low": ci_lo, "cv_ci_high": ci_hi,
        "cv_macroF1": f1_score(y_train, p_cv > 0.5, average="macro"),
        "cv_balAcc": balanced_accuracy_score(y_train, p_cv > 0.5),
        "test_auc": roc_auc_score(y_test, p_test), "test_ci_low": ci_lo_t, "test_ci_high": ci_hi_t,
        "test_macroF1": f1_score(y_test, p_test > 0.5, average="macro"),
        "test_balAcc": balanced_accuracy_score(y_test, p_test > 0.5),
    })
    print(f"{m['display_name']:24s}  CV AUC {roc_auc_score(y_train, p_cv):.4f}"
          f"  test AUC {roc_auc_score(y_test, p_test):.4f}  [{time.time() - t0:.0f}s]")

frozen_df = pd.DataFrame(FROZEN_RESULTS)
BEST_FROZEN_KEY = frozen_df.loc[frozen_df.cv_auc.idxmax(), "key"] if len(frozen_df) else None
print(f"\nbest-performing frozen encoder (by CV AUC): {BEST_FROZEN_KEY}")

C5-FERNET (frozen)        CV AUC 0.7473  test AUC 0.6692  [19s]
RobeCzech (frozen)        CV AUC 0.8176  test AUC 0.8647  [11s]
CZERT-B (frozen)          CV AUC 0.8156  test AUC 0.8296  [11s]
mBERT (frozen)            CV AUC 0.7849  test AUC 0.7318  [11s]
Small-E-Czech (frozen)    CV AUC 0.8065  test AUC 0.8747  [5s]

best-performing frozen encoder (by CV AUC): robeczech_base


## Block 7 — Fine-tuned classifier arm (gated; lighter single-5-fold protocol)

`RUN_FINETUNE=False` skips this block entirely (see Block 0). The notebook's matched
protocol everywhere else is 10× repeated stratified 5-fold (50 folds, `SEED=42`) — but
fine-tuning 5 transformer models over 50 folds each (250 fine-tune runs, plus 5 more for
the test estimate) is not practical on a single machine. This block therefore runs a
**deliberately lighter, separately-labelled protocol for this arm only**:

* **`FT_N_REPEATS=1`, `FT_N_SPLITS=5`** — a single stratified 5-fold CV (`SEED=42`), not
  the repeated version. 5 fold-fits per model instead of 50. This is still proper held-out
  CV (strictly more rigorous than the professor's single 75/25 split, which has no
  held-out generalisation estimate at all), just noisier than 10 repeats would be — the
  fold-assignment variance isn't averaged away. **Fine-tuned rows in the tables are
  labelled `(fine-tuned, 1x5-fold CV)` for exactly this reason: they are not under the
  identical protocol as the linear-probe rows next to them.**
* **241 train docs only, still no `train.extra`** — same constraint as every other arm.
  With 5-fold splits (not the same folds as `train.extra`-inclusive protocols), each
  fold's training portion is ~193 docs.

**Overfitting safeguards**, given a fine-tuned 100M+ parameter encoder is being fit on an
already-small ~193-document fold, further split for early stopping:

* **80/20 fold-internal train/val split** (not 85/15) for early-stopping — the smaller
  training slice buys a more reliable stopping signal on the ~39 validation docs, which
  matters more than the marginal training-data loss at this scale.
* **Early stopping on fold-internal validation loss**, patience 2 epochs, best-weights
  restored — the fold's *held-out* validation rows (used for the OOF prediction) are never
  part of this internal split and never influence training or model selection.
* **Explicit weight decay (0.01)** on `AdamW`, gradient-norm clipping at 1.0 (both already
  implicit/present, made explicit here given the overfitting risk).
* **Per-fold logging** of the stopping epoch and best fold-internal validation loss, so
  overfitting (a val loss that never improves past epoch 1, or a training loss that keeps
  dropping while val loss rises) is visible in the output rather than silent.

Otherwise unchanged from the original notebook's hyperparameters: `AdamW`, linear warmup,
`MAX_LENGTH=256`, `BATCH_SIZE=8`, `LEARNING_RATE=1e-5`, `WARMUP_RATIO=0.1`, up to 8 epochs.
Fine-tuned OOF/test predictions are cached to disk, like the embeddings in Block 5.

**Process isolation (found necessary on this machine, kept as a general safeguard).**
Sequentially fine-tuning two *different* transformer architectures inside one Python
process was found to reliably hang the MPS backend on this machine — confirmed by an
isolated repro: the second model's backward pass never returns; the process sits in an
uninterruptible kernel wait indefinitely. This is not a timing/slowness issue and was not
fixed by explicit `gc.collect()` / `torch.mps.empty_cache()` between models. The mitigation
is `_ft_worker.py` (next to this notebook): each model's full single-5-fold run happens in
its **own subprocess**, invoked once per model, guaranteeing a clean device context per
model regardless of backend (MPS or CUDA). On this machine (~8GB RAM), even one isolated
subprocess fine-tuning a ~110–180M-parameter encoder was slow enough that a full run was
not completed here — **this arm is verified to launch correctly but has not been run to
completion on this machine.** To actually run it: use a machine with a CUDA GPU (or more
headroom), set `RUN_FINETUNE=True` in Block 0, and re-run this notebook — the same
`_ft_worker.py` subprocess call works unchanged there.

In [10]:
import subprocess, sys

PROBA_FT, TEST_PROBA_FT, FT_RESULTS, FT_DISPLAY_NAME = {}, {}, [], {}
FT_N_REPEATS, FT_N_SPLITS = 1, 5   # single 5-fold CV for THIS ARM ONLY -- see Block 7 markdown.
FT_PROTOCOL_TAG = f"(fine-tuned, {FT_N_REPEATS}x{FT_N_SPLITS}-fold CV)"
FT_WORKER = Path("_ft_worker.py")   # runs each model in its own subprocess -- see Block 7 markdown

if not RUN_FINETUNE:
    print("RUN_FINETUNE=False -- skipping the fine-tuned classifier arm entirely (Block 0).")
elif not HAS_ACCELERATOR:
    print("No GPU/MPS accelerator available -- skipping the fine-tuned classifier arm "
          "(fine-tuning transformer models without an accelerator is not practical).")
else:
    for m in MODEL_CONFIGS:
        key = m["key"]
        FT_DISPLAY_NAME[key] = m["display_name"].replace("(frozen)", FT_PROTOCOL_TAG)
        cache_path = OUT / f"{key}_finetuned_oof.npz"
        if not cache_path.exists():
            print(f"\nfine-tuning {FT_DISPLAY_NAME[key]} in a fresh subprocess "
                  f"({FT_N_REPEATS * FT_N_SPLITS} fold fits + 1 test fit)...")
            t0 = time.time()
            proc = subprocess.run([sys.executable, str(FT_WORKER), key],
                                  capture_output=True, text=True)
            print(proc.stdout)
            if proc.returncode != 0:
                print(f"  {key}: subprocess FAILED (exit {proc.returncode}) -- skipping this model")
                print(proc.stderr[-2000:])
                continue
            print(f"  [{time.time() - t0:.0f}s]")
        if not cache_path.exists():
            continue

        _c = np.load(cache_path)
        p_cv, p_test = _c["cv_proba"], _c["test_proba"]
        # alignment guard -- the worker rebuilds y_train/y_test itself; they must match
        # this notebook's own arrays exactly, or the OOF predictions are not usable for
        # paired bootstrap against the other arms.
        assert np.array_equal(_c["y_train"], y_train) and np.array_equal(_c["y_test"], y_test), \
            f"{key}: worker's doc order/labels do not match this notebook's -- do not use"

        ci_lo, ci_hi = boot_ci(y_train, p_cv, seed=SEED)
        ci_lo_t, ci_hi_t = boot_ci(y_test, p_test, seed=SEED)
        PROBA_FT[key] = p_cv
        TEST_PROBA_FT[key] = p_test
        FT_RESULTS.append({
            "key": key + "_ft", "representation": FT_DISPLAY_NAME[key],
            "type": "embedding-finetuned", "n_feat": "—",
            "cv_auc": roc_auc_score(y_train, p_cv), "cv_ci_low": ci_lo, "cv_ci_high": ci_hi,
            "cv_macroF1": f1_score(y_train, p_cv > 0.5, average="macro"),
            "cv_balAcc": balanced_accuracy_score(y_train, p_cv > 0.5),
            "test_auc": roc_auc_score(y_test, p_test), "test_ci_low": ci_lo_t, "test_ci_high": ci_hi_t,
            "test_macroF1": f1_score(y_test, p_test > 0.5, average="macro"),
            "test_balAcc": balanced_accuracy_score(y_test, p_test > 0.5),
        })
        print(f"  {FT_DISPLAY_NAME[key]}: CV AUC {roc_auc_score(y_train, p_cv):.4f}  "
              f"test AUC {roc_auc_score(y_test, p_test):.4f}")

ft_df = pd.DataFrame(FT_RESULTS)


fine-tuning C5-FERNET (fine-tuned, 1x5-fold CV) in a fresh subprocess (5 fold fits + 1 test fit)...


Exception in thread Thread-5 (_readerthread):
Traceback (most recent call last):
  File "C:\Users\solarisflos\AppData\Local\Python\pythoncore-3.14-64\Lib\threading.py", line 1082, in _bootstrap_inner
    self._context.run(self.run)
    ~~~~~~~~~~~~~~~~~^^^^^^^^^^
  File "C:\Users\solarisflos\AppData\Local\Python\pythoncore-3.14-64\Lib\threading.py", line 1024, in run
    self._target(*self._args, **self._kwargs)
    ~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\solarisflos\AppData\Local\Python\pythoncore-3.14-64\Lib\subprocess.py", line 1613, in _readerthread
    buffer.append(fh.read())
                  ~~~~~~~^^
  File "C:\Users\solarisflos\AppData\Local\Python\pythoncore-3.14-64\Lib\encodings\cp1252.py", line 23, in decode
    return codecs.charmap_decode(input,self.errors,decoding_table)[0]
           ~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
UnicodeDecodeError: 'charmap' codec can't decode byte 0x8f in position 4382: character maps to <undefined>


[fernet_c5_roberta] device=cuda  hf=fav-kky/FERNET-C5-RoBERTa  protocol=1x5-fold CV
    fold 1/5: stopped at epoch 8/8, best val_loss=0.3735
    fold 2/5: stopped at epoch 4/8, best val_loss=0.5246
    fold 3/5: stopped at epoch 4/8, best val_loss=0.5401
    fold 4/5: stopped at epoch 8/8, best val_loss=0.4678
    fold 5/5: stopped at epoch 4/8, best val_loss=0.5726
    test fit: stopped at epoch 8/8, best val_loss=0.3116
[fernet_c5_roberta] wrote fileDataset\outputs\fernet_c5_roberta_finetuned_oof.npz

  [149s]
  C5-FERNET (fine-tuned, 1x5-fold CV): CV AUC 0.7853  test AUC 0.7306

fine-tuning RobeCzech (fine-tuned, 1x5-fold CV) in a fresh subprocess (5 fold fits + 1 test fit)...


Exception in thread Thread-7 (_readerthread):
Traceback (most recent call last):
  File "C:\Users\solarisflos\AppData\Local\Python\pythoncore-3.14-64\Lib\threading.py", line 1082, in _bootstrap_inner
    self._context.run(self.run)
    ~~~~~~~~~~~~~~~~~^^^^^^^^^^
  File "C:\Users\solarisflos\AppData\Local\Python\pythoncore-3.14-64\Lib\threading.py", line 1024, in run
    self._target(*self._args, **self._kwargs)
    ~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\solarisflos\AppData\Local\Python\pythoncore-3.14-64\Lib\subprocess.py", line 1613, in _readerthread
    buffer.append(fh.read())
                  ~~~~~~~^^
  File "C:\Users\solarisflos\AppData\Local\Python\pythoncore-3.14-64\Lib\encodings\cp1252.py", line 23, in decode
    return codecs.charmap_decode(input,self.errors,decoding_table)[0]
           ~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
UnicodeDecodeError: 'charmap' codec can't decode byte 0x8d in position 5092: character maps to <undefined>


[robeczech_base] device=cuda  hf=ufal/robeczech-base  protocol=1x5-fold CV
    fold 1/5: stopped at epoch 8/8, best val_loss=0.3590
    fold 2/5: stopped at epoch 8/8, best val_loss=0.3455
    fold 3/5: stopped at epoch 8/8, best val_loss=0.4463
    fold 4/5: stopped at epoch 7/8, best val_loss=0.4896
    fold 5/5: stopped at epoch 4/8, best val_loss=0.5851
    test fit: stopped at epoch 7/8, best val_loss=0.3948
[robeczech_base] wrote fileDataset\outputs\robeczech_base_finetuned_oof.npz

  [186s]
  RobeCzech (fine-tuned, 1x5-fold CV): CV AUC 0.7762  test AUC 0.8308

fine-tuning CZERT-B (fine-tuned, 1x5-fold CV) in a fresh subprocess (5 fold fits + 1 test fit)...
[czert_b_base_cased] device=cuda  hf=UWB-AIR/Czert-B-base-cased  protocol=1x5-fold CV
    fold 1/5: stopped at epoch 8/8, best val_loss=0.3231
    fold 2/5: stopped at epoch 8/8, best val_loss=0.3322
    fold 3/5: stopped at epoch 4/8, best val_loss=0.3504
    fold 4/5: stopped at epoch 7/8, best val_loss=0.3797
    fold 5/5: 

Exception in thread Thread-11 (_readerthread):
Traceback (most recent call last):
  File "C:\Users\solarisflos\AppData\Local\Python\pythoncore-3.14-64\Lib\threading.py", line 1082, in _bootstrap_inner
    self._context.run(self.run)
    ~~~~~~~~~~~~~~~~~^^^^^^^^^^
  File "C:\Users\solarisflos\AppData\Local\Python\pythoncore-3.14-64\Lib\threading.py", line 1024, in run
    self._target(*self._args, **self._kwargs)
    ~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\solarisflos\AppData\Local\Python\pythoncore-3.14-64\Lib\subprocess.py", line 1613, in _readerthread
    buffer.append(fh.read())
                  ~~~~~~~^^
  File "C:\Users\solarisflos\AppData\Local\Python\pythoncore-3.14-64\Lib\encodings\cp1252.py", line 23, in decode
    return codecs.charmap_decode(input,self.errors,decoding_table)[0]
           ~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
UnicodeDecodeError: 'charmap' codec can't decode byte 0x8d in position 4453: character maps to <undefined>


[mbert_base_cased] device=cuda  hf=google-bert/bert-base-multilingual-cased  protocol=1x5-fold CV
    fold 1/5: stopped at epoch 6/8, best val_loss=0.3772
    fold 2/5: stopped at epoch 8/8, best val_loss=0.3922
    fold 3/5: stopped at epoch 7/8, best val_loss=0.4499
    fold 4/5: stopped at epoch 7/8, best val_loss=0.4608
    fold 5/5: stopped at epoch 7/8, best val_loss=0.5573
    test fit: stopped at epoch 8/8, best val_loss=0.3402
[mbert_base_cased] wrote fileDataset\outputs\mbert_base_cased_finetuned_oof.npz

  [250s]
  mBERT (fine-tuned, 1x5-fold CV): CV AUC 0.7914  test AUC 0.8496

fine-tuning Small-E-Czech (fine-tuned, 1x5-fold CV) in a fresh subprocess (5 fold fits + 1 test fit)...
[small_e_czech] device=cuda  hf=Seznam/small-e-czech  protocol=1x5-fold CV
    fold 1/5: stopped at epoch 8/8, best val_loss=0.6413
    fold 2/5: stopped at epoch 8/8, best val_loss=0.6447
    fold 3/5: stopped at epoch 8/8, best val_loss=0.6449
    fold 4/5: stopped at epoch 8/8, best val_loss=0.6

## Block 8 — Paired bootstrap contrasts (CV)

`paired_bootstrap` on out-of-fold probabilities that all share the **same** fold
assignment (verified in Block 3). For each frozen encoder: vs surface stats, vs the V2
frozen primary. For the best-performing frozen encoder: vs TF-IDF char 3–5-gram. Same
contrasts repeated for fine-tuned models when `RUN_FINETUNE=True`.

In [11]:
def paired(a, b, y=y_train, n=2000):
    return paired_bootstrap(a, b, y, n=n, seed=SEED)

CV_CONTRASTS = []
for m in MODEL_CONFIGS:
    key = m["key"]
    if key not in PROBA_FROZEN:
        continue
    d, ci, pneg = paired(PROBA_FROZEN[key], p_surf)
    CV_CONTRASTS.append({"contrast": f"{m['display_name']} vs surface stats",
                         "delta_auc": d, "ci_low": ci[0], "ci_high": ci[1], "p_delta_le_0": pneg})
    d, ci, pneg = paired(PROBA_FROZEN[key], p_v2fp)
    CV_CONTRASTS.append({"contrast": f"{m['display_name']} vs V2 frozen primary",
                         "delta_auc": d, "ci_low": ci[0], "ci_high": ci[1], "p_delta_le_0": pneg})

if BEST_FROZEN_KEY is not None:
    best_name = next(m["display_name"] for m in MODEL_CONFIGS if m["key"] == BEST_FROZEN_KEY)
    d, ci, pneg = paired(PROBA_FROZEN[BEST_FROZEN_KEY], p_tfidf)
    CV_CONTRASTS.append({"contrast": f"{best_name} (best frozen) vs TF-IDF char 3-5-gram",
                         "delta_auc": d, "ci_low": ci[0], "ci_high": ci[1], "p_delta_le_0": pneg})

for m in MODEL_CONFIGS:
    key = m["key"]
    if key not in PROBA_FT:
        continue
    ft_name = FT_DISPLAY_NAME[key]
    d, ci, pneg = paired(PROBA_FT[key], p_surf)
    CV_CONTRASTS.append({"contrast": f"{ft_name} vs surface stats",
                         "delta_auc": d, "ci_low": ci[0], "ci_high": ci[1], "p_delta_le_0": pneg})
    d, ci, pneg = paired(PROBA_FT[key], p_v2fp)
    CV_CONTRASTS.append({"contrast": f"{ft_name} vs V2 frozen primary",
                         "delta_auc": d, "ci_low": ci[0], "ci_high": ci[1], "p_delta_le_0": pneg})

cv_contrasts_df = pd.DataFrame(CV_CONTRASTS)
print(f"{'contrast':52s} {'dAUC':>8s}  {'95% CI':>18s}  {'P(d<=0)':>8s}")
for _, r in cv_contrasts_df.iterrows():
    flag = "  *" if (r.ci_low > 0 or r.ci_high < 0) else ""
    print(f"{r.contrast:52s} {r.delta_auc:+.4f}  [{r.ci_low:+.4f},{r.ci_high:+.4f}]  {r.p_delta_le_0:6.3f}{flag}")

contrast                                                 dAUC              95% CI   P(d<=0)
C5-FERNET (frozen) vs surface stats                  +0.0042  [-0.0643,+0.0755]   0.465
C5-FERNET (frozen) vs V2 frozen primary              -0.0593  [-0.1282,+0.0124]   0.952
RobeCzech (frozen) vs surface stats                  +0.0755  [+0.0041,+0.1439]   0.018  *
RobeCzech (frozen) vs V2 frozen primary              +0.0120  [-0.0524,+0.0774]   0.358
CZERT-B (frozen) vs surface stats                    +0.0728  [+0.0039,+0.1447]   0.021  *
CZERT-B (frozen) vs V2 frozen primary                +0.0093  [-0.0487,+0.0674]   0.381
mBERT (frozen) vs surface stats                      +0.0425  [-0.0315,+0.1112]   0.117
mBERT (frozen) vs V2 frozen primary                  -0.0210  [-0.0804,+0.0372]   0.761
Small-E-Czech (frozen) vs surface stats              +0.0645  [-0.0040,+0.1290]   0.033
Small-E-Czech (frozen) vs V2 frozen primary          +0.0010  [-0.0540,+0.0555]   0.479
RobeCzech (frozen) (be

## Block 9 — Paired bootstrap contrasts (test, one-shot)

Same contrasts on the 61-document test predictions: each model vs the frozen V2 primary
pipeline and vs the frozen surface pipeline (both loaded in Block 3, no refitting). With
n=61 these CIs are wide — an honest reflection of sample size, same caveat as notebook 10.

In [12]:
TEST_CONTRASTS = []
for m in MODEL_CONFIGS:
    key = m["key"]
    if key not in TEST_PROBA_FROZEN:
        continue
    d, ci, pneg = paired_bootstrap(TEST_PROBA_FROZEN[key], proba_v2fp_test, y_test, n=2000, seed=SEED)
    TEST_CONTRASTS.append({"contrast": f"{m['display_name']} vs V2 frozen primary (test)",
                           "delta_auc": d, "ci_low": ci[0], "ci_high": ci[1], "p_delta_le_0": pneg})
    d, ci, pneg = paired_bootstrap(TEST_PROBA_FROZEN[key], proba_surface_test, y_test, n=2000, seed=SEED)
    TEST_CONTRASTS.append({"contrast": f"{m['display_name']} vs surface (test)",
                           "delta_auc": d, "ci_low": ci[0], "ci_high": ci[1], "p_delta_le_0": pneg})

for m in MODEL_CONFIGS:
    key = m["key"]
    if key not in TEST_PROBA_FT:
        continue
    ft_name = FT_DISPLAY_NAME[key]
    d, ci, pneg = paired_bootstrap(TEST_PROBA_FT[key], proba_v2fp_test, y_test, n=2000, seed=SEED)
    TEST_CONTRASTS.append({"contrast": f"{ft_name} vs V2 frozen primary (test)",
                           "delta_auc": d, "ci_low": ci[0], "ci_high": ci[1], "p_delta_le_0": pneg})
    d, ci, pneg = paired_bootstrap(TEST_PROBA_FT[key], proba_surface_test, y_test, n=2000, seed=SEED)
    TEST_CONTRASTS.append({"contrast": f"{ft_name} vs surface (test)",
                           "delta_auc": d, "ci_low": ci[0], "ci_high": ci[1], "p_delta_le_0": pneg})

test_contrasts_df = pd.DataFrame(TEST_CONTRASTS)
print(f"{'contrast':46s} {'dAUC':>8s}  {'95% CI':>18s}  {'P(d<=0)':>8s}")
for _, r in test_contrasts_df.iterrows():
    flag = "  *" if (r.ci_low > 0 or r.ci_high < 0) else ""
    print(f"{r.contrast:46s} {r.delta_auc:+.4f}  [{r.ci_low:+.4f},{r.ci_high:+.4f}]  {r.p_delta_le_0:6.3f}{flag}")

contrast                                           dAUC              95% CI   P(d<=0)
C5-FERNET (frozen) vs V2 frozen primary (test) -0.1360  [-0.2752,+0.0035]   0.972
C5-FERNET (frozen) vs surface (test)           -0.1251  [-0.2584,-0.0013]   0.977  *
RobeCzech (frozen) vs V2 frozen primary (test) +0.0569  [-0.0536,+0.1778]   0.153
RobeCzech (frozen) vs surface (test)           +0.0677  [-0.0153,+0.1625]   0.068
CZERT-B (frozen) vs V2 frozen primary (test)   +0.0212  [-0.0890,+0.1234]   0.327
CZERT-B (frozen) vs surface (test)             +0.0321  [-0.0631,+0.1404]   0.274
mBERT (frozen) vs V2 frozen primary (test)     -0.0756  [-0.2080,+0.0504]   0.857
mBERT (frozen) vs surface (test)               -0.0647  [-0.1893,+0.0571]   0.855
Small-E-Czech (frozen) vs V2 frozen primary (test) +0.0667  [-0.0381,+0.1739]   0.097
Small-E-Czech (frozen) vs surface (test)       +0.0776  [-0.0178,+0.1917]   0.064
C5-FERNET (fine-tuned, 1x5-fold CV) vs V2 frozen primary (test) -0.0764  [-0.2072,+0.05

## Block 10 — Final tables + save

**Table 1** rows for transcript length / surface stats / V1 categorical / V2 stable+surface
/ TF-IDF come straight from `frozen_spec.json` and `v2_stability_comparison.csv` (no
recomputation — "(spec)" in the printed table). Embedding rows come from Blocks 6–7.
**Table 2** is the concatenation of Blocks 8–9. Dr. Vita's unmatched numbers are printed
once more directly above Table 1 so a reader has both panels side by side without
mistaking one for the other.

In [13]:
_rec = pd.read_csv(OUT / "v2_stability_comparison.csv").set_index("key")
_tr  = spec["test_results"]

def _spec_row(name, rep_type, cv_key, test_auc=None, test_ci=None):
    r = _rec.loc[cv_key]
    return {"representation": name, "type": rep_type,
            "cv_auc": r.AUC, "cv_ci": f"[{r.CI_low:.3f}, {r.CI_high:.3f}]",
            "test_auc": test_auc if test_auc is not None else np.nan,
            "test_ci": (f"[{test_ci[0]:.3f}, {test_ci[1]:.3f}]" if test_ci is not None else "—")}

table1_rows = [
    _spec_row("Transcript length",    "surface", "len"),
    _spec_row("Surface stats",        "surface", "surf",
             test_auc=_tr["surface_auc"], test_ci=None),  # no persisted surface test CI
    _spec_row("V1 categorical",       "LLM",     "v1"),
    _spec_row("V2 stable+surface",    "LLM",     "v2fp",
             test_auc=_tr["primary_auc"], test_ci=tuple(_tr["primary_ci95"])),
    _spec_row("TF-IDF char 3-5-gram", "lexical", "tfidf"),
]
# surface test CI: recomputed here (boot_ci on the loaded frozen predictions, Block 3) --
# not persisted in frozen_spec.json, but no refitting is involved (predict_proba only).
_surf_test_ci = boot_ci(y_test, proba_surface_test, seed=SEED)
table1_rows[1]["test_ci"] = f"[{_surf_test_ci[0]:.3f}, {_surf_test_ci[1]:.3f}]"

for r in frozen_df.itertuples():
    table1_rows.append({"representation": r.representation, "type": "embedding",
                        "cv_auc": r.cv_auc, "cv_ci": f"[{r.cv_ci_low:.3f}, {r.cv_ci_high:.3f}]",
                        "test_auc": r.test_auc, "test_ci": f"[{r.test_ci_low:.3f}, {r.test_ci_high:.3f}]"})
if len(FT_RESULTS):
    for r in ft_df.itertuples():
        table1_rows.append({"representation": r.representation, "type": "embedding-finetuned",
                            "cv_auc": r.cv_auc, "cv_ci": f"[{r.cv_ci_low:.3f}, {r.cv_ci_high:.3f}]",
                            "test_auc": r.test_auc, "test_ci": f"[{r.test_ci_low:.3f}, {r.test_ci_high:.3f}]"})

table1 = pd.DataFrame(table1_rows)
table2 = pd.concat([cv_contrasts_df, test_contrasts_df], ignore_index=True)

print("=" * 90)
print("UNMATCHED REFERENCE (repeated) -- Dr. Vita's original notebook, 327 docs, single")
print("75/25 split, SEED=1704, no repeated CV, no bootstrap CI. NOT comparable below.")
print("=" * 90)
print(ORIGINAL_UNMATCHED.to_string(index=False))
print()
print("=" * 90)
print("TABLE 1 -- Full matched comparison (all arms, one protocol: 241 train / 61 test,")
print(f"10x5 repeated stratified CV, SEED={SEED}, 2000-resample bootstrap CI)")
print("=" * 90)
print(table1.to_string(index=False))
if RUN_FINETUNE and PROBA_FT:
    print(f"\nNOTE: rows tagged {FT_PROTOCOL_TAG} use a single stratified {FT_N_SPLITS}-fold CV"
          f" (not the 10x{N_SPLITS_DEFAULT} repeated protocol above) -- see Block 7.")
print()
print("=" * 90)
print("TABLE 2 -- Paired bootstrap contrasts (2000 resamples)")
print("=" * 90)
print(table2.to_string(index=False))

table1.to_csv(OUT / "embedding_baseline_comparison.csv", index=False)
table2.to_csv(OUT / "embedding_baseline_paired_bootstrap.csv", index=False)

results_json = {
    "created_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds"),
    "run_finetune": RUN_FINETUNE,
    "device": str(device),
    "protocol": {"n_train": len(y_train), "n_test": len(y_test),
                "n_repeats": N_REPEATS_DEFAULT, "n_splits": N_SPLITS_DEFAULT, "seed": SEED,
                "n_bootstrap": 2000},
    "finetune_protocol": ({
        "note": "deliberately lighter than the matched 10x5 protocol above -- see Block 7",
        "n_repeats": FT_N_REPEATS, "n_splits": FT_N_SPLITS, "seed": SEED,
        "n_train": len(y_train), "n_test": len(y_test),
    } if RUN_FINETUNE and PROBA_FT else None),
    "unmatched_reference": ORIGINAL_UNMATCHED.to_dict(orient="records"),
    "table1": table1.to_dict(orient="records"),
    "table2": table2.to_dict(orient="records"),
    "reproduction_check": {
        k: {"reproduced_auc": float(roc_auc_score(y_train, p)), "recorded_auc": float(_rec.loc[k, "AUC"])}
        for k, p in [("surf", p_surf), ("v2fp", p_v2fp), ("tfidf", p_tfidf)]
    },
}
(OUT / "embedding_baseline_results.json").write_text(json.dumps(results_json, indent=2, default=str))

print(f"\nsaved -> {OUT / 'embedding_baseline_comparison.csv'}")
print(f"saved -> {OUT / 'embedding_baseline_paired_bootstrap.csv'}")
print(f"saved -> {OUT / 'embedding_baseline_results.json'}")
print(f"cached embeddings -> {OUT}/<model_key>_embeddings_{{train,test}}.npy")

UNMATCHED REFERENCE (repeated) -- Dr. Vita's original notebook, 327 docs, single
75/25 split, SEED=1704, no repeated CV, no bootstrap CI. NOT comparable below.
        model                            hf_checkpoint  original_test_auc
    C5-FERNET                fav-kky/FERNET-C5-RoBERTa              0.744
    RobeCzech                      ufal/robeczech-base              0.836
      CZERT-B               UWB-AIR/Czert-B-base-cased              0.822
        mBERT google-bert/bert-base-multilingual-cased              0.798
Small-E-Czech                     Seznam/small-e-czech              0.782

TABLE 1 -- Full matched comparison (all arms, one protocol: 241 train / 61 test,
10x5 repeated stratified CV, SEED=42, 2000-resample bootstrap CI)
                         representation                type   cv_auc          cv_ci  test_auc        test_ci
                      Transcript length             surface 0.693901 [0.618, 0.766]       NaN              —
                          Surf